# N-body gravity in 3D

*mechanics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/mechanics/n_body_gravity)

Newtonian gravity between $N$ bodies. Every pairwise separation $r_{abi}$ is built at once by broadcasting, one einsum sums the pulls of all bodies $b$ on each body $a$, and velocity Verlet steps the whole system forward.

**Vector form**

$$\ddot{\mathbf r}_a = G\sum_{b\neq a} m_b\,\frac{\mathbf r_b - \mathbf r_a}{\left(\lVert\mathbf r_b - \mathbf r_a\rVert^2 + \varepsilon^2\right)^{3/2}}$$

**Index (tensor) form**

$$r_{abi} = x_{bi} - x_{ai},\qquad \ddot x_{ai} = G\,m_b\,\left(r_{abj}r_{abj} + \varepsilon^2\right)^{-3/2}\,r_{abi}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{ai}$ | `x` | (N, 3) | position of body $a$, component $i \in \{x, y, z\}$ |
| $m_b$ | `m` | (N,) | mass of body $b$ |
| $r_{abi}$ | `x[None, :, :] - x[:, None, :]` | (N, N, 3) | separation from $a$ to $b$, all pairs by broadcasting |
| $r_{abj}r_{abj}$ | `einsum('abj,abj->ab', r, r)` | (N, N) | repeated $j$ ⇒ summed: squared distance of every pair |
| $\ddot x_{ai}$ | `G * einsum('b,ab,abi->ai', m, inv_d3, r)` | (N, 3) | repeated $b$ ⇒ summed: total pull on body $a$ |
| $E$ | `energy(x, v, m, G, eps)` | () | kinetic + pairwise potential, conserved |

The softening length $\varepsilon$ keeps the force finite when two bodies pass very close; with $\varepsilon = 0$ this is exact Newtonian gravity. Units are chosen so that $G = 1$. Velocity Verlet is symplectic, so the energy error stays small and bounded instead of drifting. The figure-8 is the famous periodic three-body orbit found by Moore (1993) and Chenciner & Montgomery (2000).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/mechanics/n_body_gravity/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    G, eps = tensors.G, params["softening"]
    m, x0, v0 = tensors.initial_state(params["scenario"], dev)

    times, xs, vs = solve.solve(m, x0, v0, G, eps, params["t_max"], params["dt"])

    e_start = equation.energy(xs[0], vs[0], m, G, eps).item()
    e_end = equation.energy(xs[-1], vs[-1], m, G, eps).item()
    p_end = equation.momentum(vs[-1], m).norm().item()
    print(f"{len(m)} bodies, {len(times) - 1} steps")
    print(f"energy {e_start:.6f} -> {e_end:.6f}  (relative change {abs(e_end - e_start) / abs(e_start):.2e})")
    print(f"total momentum at the end |P| = {p_end:.2e}")
    return plot.animate(times, xs, m, out_dir / "n_body_gravity.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/mechanics/n_body_gravity/equation.py
r"""Physics → tensor translation.

Newton's law of gravitation for N bodies (with a small softening length ε),

    \ddot{\mathbf r}_a = G \sum_{b \neq a} m_b
        \frac{\mathbf r_b - \mathbf r_a}{(\lVert \mathbf r_b - \mathbf r_a \rVert^2 + \varepsilon^2)^{3/2}},

written in index notation for bodies a, b and spatial components i, j:

    r_{abi} = x_{bi} - x_{ai}                                   (separation tensor, all pairs)
    \ddot x_{ai} = G \, m_b \, (r_{abj} r_{abj} + \varepsilon^2)^{-3/2} \, r_{abi}

Repeated indices are summed: j gives the squared distance of each pair, einsum('abj,abj->ab'),
and b adds up the pull of every other body on body a, einsum('b,ab,abi->ai').
The term b = a vanishes because r_{aai} = 0, so we simply zero the diagonal.

Conserved quantities: total momentum P_i = m_a v_{ai} and the energy

    E = \tfrac12 m_a v_{ai} v_{ai} - \tfrac12 G \sum_{a \neq b} m_a m_b (r_{abj} r_{abj} + \varepsilon^2)^{-1/2}
"""

import torch


def separations(x: torch.Tensor) -> torch.Tensor:
    r"""r_{abi} = x_{bi} - x_{ai}.   x: (N, 3) → (N, N, 3) by broadcasting."""
    return x[None, :, :] - x[:, None, :]


def acceleration(x: torch.Tensor, m: torch.Tensor, G: float, eps: float) -> torch.Tensor:
    r"""\ddot x_{ai} = G m_b (r_{abj} r_{abj} + ε^2)^{-3/2} r_{abi}.   x: (N, 3), m: (N,) → (N, 3)"""
    r = separations(x)  # (N, N, 3)
    dist_sq = torch.einsum("abj,abj->ab", r, r) + eps**2  # (N, N)
    inv_d3 = dist_sq ** (-1.5)  # (N, N)
    inv_d3.fill_diagonal_(0.0)  # no self-force (also avoids 1/0 when ε = 0)
    return G * torch.einsum("b,ab,abi->ai", m, inv_d3, r)  # sum over b


def energy(x: torch.Tensor, v: torch.Tensor, m: torch.Tensor, G: float, eps: float) -> torch.Tensor:
    r"""E = ½ m_a v_{ai} v_{ai} - ½ G Σ_{a≠b} m_a m_b / \sqrt{r_{abj} r_{abj} + ε^2}.   x, v: (N, 3) → scalar"""
    kinetic = 0.5 * torch.einsum("a,ai,ai->", m, v, v)
    r = separations(x)
    inv_d = (torch.einsum("abj,abj->ab", r, r) + eps**2) ** (-0.5)  # (N, N)
    inv_d.fill_diagonal_(0.0)
    potential = -0.5 * G * torch.einsum("a,b,ab->", m, m, inv_d)  # ½ because each pair appears twice
    return kinetic + potential


def momentum(v: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
    r"""P_i = m_a v_{ai} (sum over bodies a).   v: (..., N, 3) → (..., 3)"""
    return torch.einsum("a,...ai->...i", m, v)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/mechanics/n_body_gravity/tensors.py
"""Build the concrete tensors: masses m_a (N,), positions x_ai (N, 3) and velocities v_ai (N, 3).

Units are chosen so that G = 1. We use float64 so that energy conservation is limited
by the integrator, not by round-off.
"""

import math

import torch

G = 1.0
DTYPE = torch.float64


def to_center_of_mass(m: torch.Tensor, x: torch.Tensor, v: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Shift so the centre of mass sits at rest at the origin: x_ai -= m_b x_bi / M (same for v)."""
    M = m.sum()
    x_com = torch.einsum("b,bi->i", m, x) / M  # (3,)
    v_com = torch.einsum("b,bi->i", m, v) / M  # (3,)
    return x - x_com, v - v_com


def figure_eight(device: torch.device | None = None) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Three equal masses chasing each other on a figure-8 (Chenciner & Montgomery). Period ≈ 6.326."""
    m = torch.ones(3, dtype=DTYPE)
    x = torch.tensor([[0.97000436, -0.24308753, 0.0], [-0.97000436, 0.24308753, 0.0], [0.0, 0.0, 0.0]], dtype=DTYPE)
    v3 = torch.tensor([-0.93240737, -0.86473146, 0.0], dtype=DTYPE)
    v = torch.stack([-v3 / 2, -v3 / 2, v3])  # (3, 3) total momentum is zero
    return m.to(device), x.to(device), v.to(device)


def star_and_planets(device: torch.device | None = None) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """A star of mass 1 with four light planets on circular, slightly tilted orbits."""
    radius = [0.6, 1.0, 1.5, 2.2]
    planet_mass = [1e-3, 3e-3, 2e-3, 1e-3]
    tilt = [math.radians(d) for d in (0.0, 6.0, -8.0, 12.0)]  # inclination about the x axis
    phase = [0.0, 2.0, 4.0, 1.0]  # starting angle on the orbit

    m, x, v = [1.0], [[0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0]]  # the star
    for r, mp, inc, th in zip(radius, planet_mass, tilt, phase):
        speed = math.sqrt(G * 1.0 / r)  # circular orbit: v² / r = G M / r²
        m.append(mp)
        x.append([r * math.cos(th), r * math.sin(th) * math.cos(inc), r * math.sin(th) * math.sin(inc)])
        v.append([-speed * math.sin(th), speed * math.cos(th) * math.cos(inc), speed * math.cos(th) * math.sin(inc)])
    m, x, v = (torch.tensor(a, dtype=DTYPE, device=device) for a in (m, x, v))
    x, v = to_center_of_mass(m, x, v)
    return m, x, v


def binary_and_planet(device: torch.device | None = None) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Two equal stars on a tight circular orbit, plus a light planet circling both on a tilted orbit."""
    m, x, v = two_body_circular(0.5, 0.5, 0.5, device)  # the binary, already in its centre-of-mass frame
    r, inc = 2.0, math.radians(20.0)
    speed = math.sqrt(G * m.sum().item() / r)  # from far away the binary pulls like one mass of 1
    planet_x = torch.tensor([[0.0, r * math.cos(inc), r * math.sin(inc)]], dtype=DTYPE, device=device)
    planet_v = torch.tensor([[-speed, 0.0, 0.0]], dtype=DTYPE, device=device)
    m = torch.cat([m, torch.tensor([1e-3], dtype=DTYPE, device=device)])  # (3,)
    x, v = torch.cat([x, planet_x]), torch.cat([v, planet_v])  # (3, 3) each
    x, v = to_center_of_mass(m, x, v)
    return m, x, v


def two_body_circular(m1: float, m2: float, separation: float, device: torch.device | None = None):
    """Two bodies on a circular orbit about their centre of mass (used to check Kepler's third law)."""
    m = torch.tensor([m1, m2], dtype=DTYPE, device=device)
    omega = math.sqrt(G * (m1 + m2) / separation**3)  # Kepler: ω² a³ = G (m1 + m2)
    x = torch.tensor([[0.0, 0.0, 0.0], [separation, 0.0, 0.0]], dtype=DTYPE, device=device)
    v = torch.tensor([[0.0, 0.0, 0.0], [0.0, omega * separation, 0.0]], dtype=DTYPE, device=device)
    x, v = to_center_of_mass(m, x, v)
    return m, x, v


def initial_state(scenario: str, device: torch.device | None = None):
    """(m (N,), x (N, 3), v (N, 3)) for the chosen scenario."""
    if scenario == "figure-8":
        return figure_eight(device)
    if scenario == "star and planets":
        return star_and_planets(device)
    return binary_and_planet(device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/mechanics/n_body_gravity/solve.py
"""Step all N bodies forward together with velocity Verlet (kick – drift – kick)."""

import torch

from .equation import acceleration


def solve(
    m: torch.Tensor, x0: torch.Tensor, v0: torch.Tensor, G: float, eps: float, t_max: float, dt: float
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Return times (T,), positions (T, N, 3) and velocities (T, N, 3).

    Velocity Verlet is symplectic: the energy error oscillates but does not drift,
    which is why it is the standard choice for orbits. One force evaluation per step.
    """
    n_steps = int(round(t_max / dt))
    x, v = x0.clone(), v0.clone()
    a = acceleration(x, m, G, eps)  # (N, 3)
    xs, vs = [x], [v]
    for _ in range(n_steps):
        v_half = v + 0.5 * dt * a  # kick
        x = x + dt * v_half  # drift
        a = acceleration(x, m, G, eps)
        v = v_half + 0.5 * dt * a  # kick
        xs.append(x)
        vs.append(v)
    times = torch.arange(n_steps + 1, device=x0.device, dtype=x0.dtype) * dt
    return times, torch.stack(xs), torch.stack(vs)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/mechanics/n_body_gravity/plot.py
"""3D animation: each body leaves a fading trail; marker size grows with mass."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(times: torch.Tensor, xs: torch.Tensor, m: torch.Tensor, path: Path, trail: float = 0.3) -> Path:
    """trail = fraction of the run that each body's tail remembers."""
    pos = xs.cpu().numpy()  # (T, N, 3)
    t = times.cpu().numpy()  # (T,)
    mass = m.cpu().numpy()  # (N,)
    tail = max(2, int(trail * len(t)))

    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(projection="3d")
    half = abs(pos).max() * 1.05
    for set_lim in (ax.set_xlim, ax.set_ylim, ax.set_zlim):
        set_lim(-half, half)
    ax.set_box_aspect((1, 1, 1))
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")

    # a body holding most of the total mass is drawn as a yellow star
    colors = ["#f59e0b" if mi > 0.5 * mass.sum() else ACCENT[a % len(ACCENT)] for a, mi in enumerate(mass)]
    sizes = 4 + 8 * (mass / mass.max()) ** (1 / 3)  # radius ∝ m^(1/3)
    trails = [ax.plot([], [], [], lw=1.2, color=c, alpha=0.7)[0] for c in colors]
    dots = [ax.plot([], [], [], "o", ms=s, color=c)[0] for s, c in zip(sizes, colors)]
    clock = ax.text2D(0.02, 0.95, "", transform=ax.transAxes)

    def update(step: int):
        start = max(0, step - tail)
        for a, (line, dot) in enumerate(zip(trails, dots)):
            path_a = pos[start : step + 1, a]  # (≤tail, 3) recent positions of body a
            line.set_data_3d(path_a[:, 0], path_a[:, 1], path_a[:, 2])
            dot.set_data_3d([pos[step, a, 0]], [pos[step, a, 1]], [pos[step, a, 2]])
        clock.set_text(f"t = {t[step]:.2f}")

    ax.view_init(elev=30, azim=-60)
    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t)), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'scenario': 'figure-8',
    't_max': 12.0,
    'dt': 0.002,
    'softening': 0.01
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.mechanics.n_body_gravity")]:
    del sys.modules[name]
main = importlib.import_module("problems.mechanics.n_body_gravity.main")
problem_dir = Path("problems/mechanics/n_body_gravity")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
